In [ ]:
# Importing Libraries
import pandas as pd
import requests
from datetime import datetime , timedelta


In [ ]:
# Load the primary CSV dataset
df= pd.read_csv(r"Tea Yield data collection.csv")

In [4]:
df.head(10)

,Year,Month,Day,DATE,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,Day Since last Pluck,NDVI
0,2020.0,10.0,5.0,10/5/2020,4.0,NaN,NaN,NaN,0,NaN
1,2020.0,10.0,17.0,10/17/2020,2.0,NaN,NaN,NaN,12,NaN
2,2020.0,11.0,2.0,11/2/2020,2.0,NaN,NaN,NaN,16,NaN
3,2020.0,11.0,9.0,11/9/2020,4.0,NaN,NaN,NaN,7,NaN
4,2020.0,11.0,20.0,11/20/2020,1.0,NaN,NaN,NaN,11,NaN
5,2020.0,11.0,28.0,11/28/2020,4.0,NaN,NaN,NaN,8,NaN
6,2020.0,12.0,8.0,12/8/2020,3.0,NaN,NaN,NaN,10,NaN
7,2020.0,12.0,18.0,12/18/2020,7.0,NaN,NaN,NaN,10,NaN
8,2020.0,12.0,27.0,12/27/2020,3.0,NaN,NaN,NaN,9,NaN
9,2021.0,1.0,4.0,1/4/2021,3.0,NaN,NaN,NaN,8,NaN


In [ ]:
# Set geographical coordinates for weather API query
# Replace with your own coordinates
LATITUDE = 7.2800 
LONGITUDE= 80.6000




In [ ]:
# Dataset columns
df.columns

Index(['Year', 'Month ', 'Day', 'DATE', 'Total yield weight per cycle (kg)',
       'Rainfall', 'Avg Temp', 'Relative humidity', 'Day Since  last Pluck',
       'NDVI'],
      dtype='object')

In [ ]:
# Standardize and rename column names for consistency
df =df.rename( columns ={'DATE':'plucking_date','Day Since  last Pluck':'plucking_interval'})

In [8]:
df.head()

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI
0,2020.0,10.0,5.0,10/5/2020,4.0,NaN,NaN,NaN,0,NaN
1,2020.0,10.0,17.0,10/17/2020,2.0,NaN,NaN,NaN,12,NaN
2,2020.0,11.0,2.0,11/2/2020,2.0,NaN,NaN,NaN,16,NaN
3,2020.0,11.0,9.0,11/9/2020,4.0,NaN,NaN,NaN,7,NaN
4,2020.0,11.0,20.0,11/20/2020,1.0,NaN,NaN,NaN,11,NaN


In [ ]:
#Convert 'plucking_date' column to datetime format
df['plucking_date'] = pd.to_datetime(df['plucking_date'])

In [ ]:
# Dataset Information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Year                               181 non-null    float64       
 1   Month                              179 non-null    float64       
 2   Day                                179 non-null    float64       
 3   plucking_date                      179 non-null    datetime64[ns]
 4   Total yield weight per cycle (kg)  179 non-null    float64       
 5   Rainfall                           0 non-null      float64       
 6   Avg Temp                           0 non-null      float64       
 7   Relative humidity                  0 non-null      float64       
 8   plucking_interval                  185 non-null    int64         
 9   NDVI                               0 non-null      float64       
dtypes: datetime64[ns](1), float64(8), int6

In [11]:
df.columns

Index(['Year', 'Month ', 'Day', 'plucking_date',
       'Total yield weight per cycle (kg)', 'Rainfall', 'Avg Temp',
       'Relative humidity', 'plucking_interval', 'NDVI'],
      dtype='object')

In [ ]:
# Initialize empty lists to store weather metrics
avg_temps=[]
total_rains=[]
avg_rains =[]
avg_humidities=[]

print ( "Data collection was started from Open - Meteo  ")

Data collection was started from Open - Meteo  


In [ ]:
# Fetch historical weather data sequentially via Open-Meteo API
avg_temps = []
total_rains = []
avg_rains = []
avg_humidities = []


for index, row in df.iterrows():
    end_date = row["plucking_date"]

    if pd.isna(end_date):
        avg_temp = total_rain = avg_rain = avg_humidity = None
    else:
        interval = row["plucking_interval"]

        if pd.isna(interval):
            interval = 0

        start_date = end_date - timedelta(days=max(int(interval) - 1, 0))

        params = {
            "latitude": LATITUDE,
            "longitude": LONGITUDE,
            "start_date": start_date.strftime("%Y-%m-%d"),
            "end_date": end_date.strftime("%Y-%m-%d"),
            "daily": (
                "temperature_2m_mean,precipitation_sum,"
                "relative_humidity_2m_mean"
            ),
            "timezone": "auto",
        }

        try:
            response = requests.get(
                "https://archive-api.open-meteo.com/v1/archive",
                params=params,
                timeout=30
            )
            response.raise_for_status()

            daily = response.json().get("daily", {})

            temps = [x for x in daily.get("temperature_2m_mean", []) if x is not None]
            rains = [x for x in daily.get("precipitation_sum", []) if x is not None]
            humidities = [
                x for x in daily.get("relative_humidity_2m_mean", [])
                if x is not None
            ]

            avg_temp = round(sum(temps) / len(temps), 2) if temps else None
            total_rain = round(sum(rains), 2) if rains else None
            avg_rain = round(sum(rains) / len(rains), 2) if rains else None
            avg_humidity = (
                round(sum(humidities) / len(humidities), 2)
                if humidities else None
            )

        except Exception as e:
            print(f"Error in row {index}: {e}")
            avg_temp = total_rain = avg_rain = avg_humidity = None

    avg_temps.append(avg_temp)
    total_rains.append(total_rain)
    avg_rains.append(avg_rain)
    avg_humidities.append(avg_humidity)

    print(f"Row {index + 1}/{len(df)} processed")

#Assign newly generated features to dataframe columns
df["Avg Temp"] = avg_temps
df["total_rain_mm"] = total_rains
df["Rainfall"] = avg_rains
df["Relative humidity"] = avg_humidities

# Save updated dataset to a new CSV file
df.to_csv("Tea Yield data collection_with_weather.csv", index=False)

print(f"Successfully saved ")

Row 1/185 processed
Row 2/185 processed
Row 3/185 processed
Row 4/185 processed
Row 5/185 processed
Row 6/185 processed
Row 7/185 processed
Row 8/185 processed
Row 9/185 processed
Row 10/185 processed
Row 11/185 processed
Row 12/185 processed
Row 13/185 processed
Row 14/185 processed
Row 15/185 processed
Row 16/185 processed
Row 17/185 processed
Row 18/185 processed
Row 19/185 processed
Row 20/185 processed
Row 21/185 processed
Row 22/185 processed
Row 23/185 processed
Row 24/185 processed
Row 25/185 processed
Row 26/185 processed
Row 27/185 processed
Row 28/185 processed
Row 29/185 processed
Row 30/185 processed
Row 31/185 processed
Row 32/185 processed
Row 33/185 processed
Row 34/185 processed
Row 35/185 processed
Row 36/185 processed
Row 37/185 processed
Row 38/185 processed
Row 39/185 processed
Row 40/185 processed
Row 41/185 processed
Row 42/185 processed
Row 43/185 processed
Row 44/185 processed
Row 45/185 processed
Row 46/185 processed
Row 47/185 processed
Row 48/185 processed
R

In [ ]:
# Display updated DataFrame with weather features
df

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
0,2020.0,10.0,5.0,2020-10-05,4.0,0.00,23.40,78.00,0,NaN,0.0
1,2020.0,10.0,17.0,2020-10-17,2.0,5.46,23.43,84.00,12,NaN,65.5
2,2020.0,11.0,2.0,2020-11-02,2.0,2.10,23.75,83.94,16,NaN,33.6
3,2020.0,11.0,9.0,2020-11-09,4.0,14.84,24.04,85.14,7,NaN,103.9
4,2020.0,11.0,20.0,2020-11-20,1.0,6.56,23.87,86.55,11,NaN,72.2
...,...,...,...,...,...,...,...,...,...,...,...
180,2026.0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,NaN,NaN
181,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,NaN,NaN
182,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,NaN,NaN
183,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,NaN,NaN


In [ ]:
# Checking for missing values
df.isna().sum()

Year                                   4
Month                                  6
Day                                    6
plucking_date                          6
Total yield weight per cycle (kg)      6
Rainfall                               6
Avg Temp                               6
Relative humidity                      6
plucking_interval                      0
NDVI                                 185
total_rain_mm                          6
dtype: int64

In [ ]:
# Checking for duplicate rows
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
180    False
181    False
182     True
183     True
184     True
Length: 185, dtype: bool

In [ ]:
#Impute missing values in 'NDVI' with a placeholder
df['NDVI']= df['NDVI'].fillna('-')

In [18]:
df.head()

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
0,2020.0,10.0,5.0,2020-10-05,4.0,0.00,23.40,78.00,0,-,0.0
1,2020.0,10.0,17.0,2020-10-17,2.0,5.46,23.43,84.00,12,-,65.5
2,2020.0,11.0,2.0,2020-11-02,2.0,2.10,23.75,83.94,16,-,33.6
3,2020.0,11.0,9.0,2020-11-09,4.0,14.84,24.04,85.14,7,-,103.9
4,2020.0,11.0,20.0,2020-11-20,1.0,6.56,23.87,86.55,11,-,72.2


In [ ]:
# Checking for missing values
df.isna().sum()

Year                                 4
Month                                6
Day                                  6
plucking_date                        6
Total yield weight per cycle (kg)    6
Rainfall                             6
Avg Temp                             6
Relative humidity                    6
plucking_interval                    0
NDVI                                 0
total_rain_mm                        6
dtype: int64

In [ ]:
# Inspect the last 7 rows of the DataFrame
df.tail(7)

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
178,2026.0,9.0,8.0,2026-09-08,19.0,2.92,25.06,78.64,11,-,32.1
179,2026.0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,-46273,-,NaN
180,2026.0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,-,NaN
181,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,-,NaN
182,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,-,NaN
183,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,-,NaN
184,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,0,-,NaN


In [ ]:
# dropping rows with missing values
df= df.dropna()

In [22]:
df.tail(10)

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
169,2026.0,5.0,26.0,2026-05-26,26.0,2.85,24.93,83.08,12,-,34.2
170,2026.0,6.0,4.0,2026-06-04,21.0,3.63,24.96,81.33,9,-,32.7
171,2026.0,6.0,15.0,2026-06-15,18.0,5.60,24.45,85.00,11,-,61.6
172,2026.0,6.0,25.0,2026-06-25,19.0,4.79,24.27,83.90,10,-,47.9
173,2026.0,7.0,6.0,2026-07-06,22.0,4.39,24.21,81.73,11,-,48.3
174,2026.0,7.0,21.0,2026-07-21,21.0,1.31,24.85,78.20,15,-,19.6
175,2026.0,8.0,1.0,2026-08-01,18.0,2.38,25.25,76.55,11,-,26.2
176,2026.0,8.0,13.0,2026-08-13,17.0,8.52,23.79,84.83,12,-,102.2
177,2026.0,8.0,28.0,2026-08-28,27.0,2.29,24.58,79.40,15,-,34.4
178,2026.0,9.0,8.0,2026-09-08,19.0,2.92,25.06,78.64,11,-,32.1


In [ ]:
# Reset index after dropping rows
df= df.reset_index()

In [24]:
df

,index,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
0,0,2020.0,10.0,5.0,2020-10-05,4.0,0.00,23.40,78.00,0,-,0.0
1,1,2020.0,10.0,17.0,2020-10-17,2.0,5.46,23.43,84.00,12,-,65.5
2,2,2020.0,11.0,2.0,2020-11-02,2.0,2.10,23.75,83.94,16,-,33.6
3,3,2020.0,11.0,9.0,2020-11-09,4.0,14.84,24.04,85.14,7,-,103.9
4,4,2020.0,11.0,20.0,2020-11-20,1.0,6.56,23.87,86.55,11,-,72.2
...,...,...,...,...,...,...,...,...,...,...,...,...
174,174,2026.0,7.0,21.0,2026-07-21,21.0,1.31,24.85,78.20,15,-,19.6
175,175,2026.0,8.0,1.0,2026-08-01,18.0,2.38,25.25,76.55,11,-,26.2
176,176,2026.0,8.0,13.0,2026-08-13,17.0,8.52,23.79,84.83,12,-,102.2
177,177,2026.0,8.0,28.0,2026-08-28,27.0,2.29,24.58,79.40,15,-,34.4


In [ ]:
# Drop redundant old index column
df.drop( columns="index", inplace = True)

In [26]:
df

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
0,2020.0,10.0,5.0,2020-10-05,4.0,0.00,23.40,78.00,0,-,0.0
1,2020.0,10.0,17.0,2020-10-17,2.0,5.46,23.43,84.00,12,-,65.5
2,2020.0,11.0,2.0,2020-11-02,2.0,2.10,23.75,83.94,16,-,33.6
3,2020.0,11.0,9.0,2020-11-09,4.0,14.84,24.04,85.14,7,-,103.9
4,2020.0,11.0,20.0,2020-11-20,1.0,6.56,23.87,86.55,11,-,72.2
...,...,...,...,...,...,...,...,...,...,...,...
174,2026.0,7.0,21.0,2026-07-21,21.0,1.31,24.85,78.20,15,-,19.6
175,2026.0,8.0,1.0,2026-08-01,18.0,2.38,25.25,76.55,11,-,26.2
176,2026.0,8.0,13.0,2026-08-13,17.0,8.52,23.79,84.83,12,-,102.2
177,2026.0,8.0,28.0,2026-08-28,27.0,2.29,24.58,79.40,15,-,34.4


In [ ]:
# Checking data types of each column 
df.dtypes

Year                                        float64
Month                                       float64
Day                                         float64
plucking_date                        datetime64[ns]
Total yield weight per cycle (kg)           float64
Rainfall                                    float64
Avg Temp                                    float64
Relative humidity                           float64
plucking_interval                             int64
NDVI                                         object
total_rain_mm                               float64
dtype: object

In [ ]:
# Converting 'Year' and 'Day' columns to integer data type
df['Year'] = df['Year'].astype(int)
df['Day'] = df['Day'].astype(int)

In [29]:
df.head()

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,NDVI,total_rain_mm
0,2020,10.0,5,2020-10-05,4.0,0.00,23.40,78.00,0,-,0.0
1,2020,10.0,17,2020-10-17,2.0,5.46,23.43,84.00,12,-,65.5
2,2020,11.0,2,2020-11-02,2.0,2.10,23.75,83.94,16,-,33.6
3,2020,11.0,9,2020-11-09,4.0,14.84,24.04,85.14,7,-,103.9
4,2020,11.0,20,2020-11-20,1.0,6.56,23.87,86.55,11,-,72.2


In [ ]:
# Rename 'Month ' to 'Month'
df= df.rename(columns= {"Month ":'Month'})

In [ ]:
# Converting 'Month' column to integer data type
df['Month'] = df["Month"].astype(int)

In [ ]:
df.rename(columns= {'plucking_date':'Plucking_date','plucking_interval':'Plucking_interval','total_rain_mm':'Total_rain_mm'}, inpalce =True)

,Year,Month,Day,Plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,Plucking_interval,NDVI,Total_rain_mm
0,2020,10,5,2020-10-05,4.0,0.00,23.40,78.00,0,-,0.0
1,2020,10,17,2020-10-17,2.0,5.46,23.43,84.00,12,-,65.5
2,2020,11,2,2020-11-02,2.0,2.10,23.75,83.94,16,-,33.6
3,2020,11,9,2020-11-09,4.0,14.84,24.04,85.14,7,-,103.9
4,2020,11,20,2020-11-20,1.0,6.56,23.87,86.55,11,-,72.2
...,...,...,...,...,...,...,...,...,...,...,...
174,2026,7,21,2026-07-21,21.0,1.31,24.85,78.20,15,-,19.6
175,2026,8,1,2026-08-01,18.0,2.38,25.25,76.55,11,-,26.2
176,2026,8,13,2026-08-13,17.0,8.52,23.79,84.83,12,-,102.2
177,2026,8,28,2026-08-28,27.0,2.29,24.58,79.40,15,-,34.4


In [ ]:
# Save final cleaned dataset to a CSV file
df.to_csv(r"Tea Yield data collection_with_weather _1Cleaned.csv",index =False)